# Pandas Recap: From Rows to Analytics

This small notebook prepares you for `062_Ecomm_Analytics.ipynb`. We use only 12 predictable e-commerce rows to practise inspection, selection, filtering, missing values, derived columns, grouping, aggregation, and ranking. Run the cells from top to bottom. The original `df` remains available so we can compare raw transactions with valid sales.

## 1. Create a tiny dataset

`StringIO` makes a multiline string behave like a file. This lets everyone use exactly the same data without downloading anything. The sample intentionally contains repeated invoices, repeated products, a missing customer ID, a cancellation, and a zero-priced damaged-stock adjustment.

In [ ]:
from io import StringIO
import pandas as pd

csv_data = """InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
1001,A100,NOTEBOOK,2,2024-01-02 09:00,5.00,101,India
1001,B200,BLUE PEN,3,2024-01-02 09:00,1.50,101,India
1002,A100,NOTEBOOK,1,2024-01-03 10:15,5.00,102,France
1002,C300,COFFEE MUG,2,2024-01-03 10:15,7.50,102,France
1003,B200,BLUE PEN,10,2024-01-04 11:30,1.50,,United Kingdom
1004,D400,CANVAS BAG,1,2024-01-05 14:00,20.00,103,India
1004,B200,BLUE PEN,5,2024-01-05 14:00,1.50,103,India
1005,A100,NOTEBOOK,4,2024-01-06 09:45,5.00,101,India
C1006,C300,COFFEE MUG,-1,2024-01-07 16:00,7.50,102,France
1007,D400,CANVAS BAG,2,2024-01-08 12:00,20.00,104,United Kingdom
1007,C300,COFFEE MUG,1,2024-01-08 12:00,7.50,104,United Kingdom
ADJ001,X999,DAMAGED STOCK,-3,2024-01-09 08:30,0.00,,United Kingdom
"""

schema = {
    "InvoiceNo": "string", "StockCode": "string",
    "Description": "string", "Quantity": "int64",
    "UnitPrice": "float64", "CustomerID": "Int64",
    "Country": "string",
}

df = pd.read_csv(
    StringIO(csv_data), dtype=schema,
    parse_dates=["InvoiceDate"], date_format="%Y-%m-%d %H:%M",
)
df

## 2. Understand rows, columns, and types

`shape` returns `(rows, columns)`. `head()` previews rows, while `dtypes` shows how pandas stores each column. IDs are strings or nullable integers because we label with them; dates are datetime values so time calculations work; quantity and price are numeric so arithmetic works.

In [ ]:
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
display(df.head(3))
df.dtypes.to_frame("Data type")

### What is a pandas Series?

A Series is a one-dimensional collection of values with labels called an index. It is similar to one spreadsheet column, but its labels do not have to be row numbers. A Series also has a name and one data type. Selecting one DataFrame column returns a Series; selecting multiple columns returns a DataFrame.

In [ ]:
daily_sales = pd.Series(
    [120.0, 175.5, 90.0],
    index=["Monday", "Tuesday", "Wednesday"],
    name="SalesAmount",
)

print("Type:", type(daily_sales).__name__)
print("Name:", daily_sales.name)
print("Data type:", daily_sales.dtype)
print("Tuesday's sales:", daily_sales.loc["Tuesday"])
daily_sales

Series operations are vectorized: arithmetic is applied to every value without a loop. The index stays attached to the results. In the second example, `df["Quantity"]` is a Series taken from the DataFrame, whereas `df[["Quantity"]]` is a one-column DataFrame.

In [ ]:
sales_after_ten_percent_growth = daily_sales * 1.10
quantity_series = df["Quantity"]
quantity_dataframe = df[["Quantity"]]

print("One bracket:", type(quantity_series).__name__, quantity_series.shape)
print("Double brackets:", type(quantity_dataframe).__name__, quantity_dataframe.shape)
sales_after_ten_percent_growth.to_frame("ProjectedSales")

## 3. Select columns and filter rows

Use one pair of brackets for a Series and a list inside double brackets for a DataFrame. A condition such as `Quantity < 0` creates one True/False value per row. Passing that Boolean Series inside `df.loc[...]` keeps only matching rows.

In [ ]:
selected_columns = df[["InvoiceNo", "StockCode", "Quantity"]]
negative_rows = df.loc[df["Quantity"] < 0]

display(selected_columns.head())
negative_rows

## 4. Add a derived column

A derived column is calculated from existing columns. Each row's `Amount` equals quantity multiplied by unit price. Positive values represent purchases; a negative value represents reversed value. The calculation is vectorized: pandas applies it to every row without writing a loop.

In [ ]:
df["Amount"] = df["Quantity"] * df["UnitPrice"]
df[["InvoiceNo", "Quantity", "UnitPrice", "Amount"]]

### Create columns with `assign()`

`assign()` returns a new DataFrame containing additional or replaced columns; it does not change the original unless the result is saved. It is useful in a method chain because several transformations remain readable. A lambda receives the DataFrame produced so far, allowing a later assigned column to use an earlier one.

In [ ]:
sales_with_features = df.assign(
    IsPositiveSale=lambda data: data["Quantity"].gt(0),
    AmountWithTax=lambda data: data["Amount"] * 1.18,
)

print("New DataFrame columns:", sales_with_features.columns.tolist())
print("Original has AmountWithTax:", "AmountWithTax" in df.columns)
sales_with_features[["InvoiceNo", "Amount", "IsPositiveSale", "AmountWithTax"]].head()

## 5. Check missing values

`isna()` marks missing cells and `sum()` counts them by column. A missing customer does not make the product sale unusable for every question. We exclude it only from customer-level analysis because we cannot assign that purchase to a known person.

In [ ]:
missing_summary = pd.DataFrame({
    "MissingCount": df.isna().sum(),
    "MissingPercent": df.isna().mean() * 100,
})
missing_summary

### Find duplicates and understand `keep`

`duplicated()` returns True/False for each row. `keep="first"` keeps the first occurrence unflagged and marks later copies. `keep="last"` marks earlier copies and leaves the last unflagged. `keep=False` marks every occurrence in a duplicate group. Use `subset` when duplication should be judged using selected business-key columns rather than the entire row.

In [ ]:
duplicate_example = pd.DataFrame([
    {"InvoiceNo": "A1", "StockCode": "PEN", "Quantity": 2},
    {"InvoiceNo": "A1", "StockCode": "BOOK", "Quantity": 1},
    {"InvoiceNo": "A1", "StockCode": "PEN", "Quantity": 2},
    {"InvoiceNo": "A2", "StockCode": "MUG", "Quantity": 1},
    {"InvoiceNo": "A1", "StockCode": "PEN", "Quantity": 2},
])

duplicate_example = duplicate_example.assign(
    KeepFirst=duplicate_example.duplicated(keep="first"),
    KeepLast=duplicate_example.duplicated(keep="last"),
    MarkAll=duplicate_example.duplicated(keep=False),
)
duplicate_example

In the result, False means “this row is kept as the representative” for `KeepFirst` or `KeepLast`; True means it is identified as a duplicate. With `MarkAll`, all three repeated `A1/PEN/2` rows are True. To remove later exact copies, use `drop_duplicates(keep="first")`. Always inspect duplicates before deleting them because identical-looking purchase lines can be legitimate.

## 6. Explore categories

`unique()` returns the category labels, `nunique()` counts them, and `value_counts()` counts their rows. These methods quickly reveal dominant or unexpected categories before grouping them into reports.

In [ ]:
print("Countries:", sorted(df["Country"].unique()))
print("Number of countries:", df["Country"].nunique())
df["Country"].value_counts().to_frame("RowCount")

### What does `nunique()` do?

`nunique()` returns the number of distinct values, not the number of rows. Repeated values count once. Missing values are excluded by default; use `dropna=False` to count missing as one additional distinct value. This is useful for questions such as “how many countries, customers, invoices, or products are represented?”

In [ ]:
customer_example = pd.Series([101, 101, 102, pd.NA], dtype="Int64")

print("Values:", customer_example.tolist())
print("Row count:", customer_example.size)
print("Unique non-missing customers:", customer_example.nunique())
print("Distinct values including missing:", customer_example.nunique(dropna=False))

df[["InvoiceNo", "StockCode", "CustomerID", "Country"]].nunique().to_frame("UniqueCount")

### What does `to_frame()` do?

A Series is one-dimensional and normally has an index plus values. `to_frame()` converts it into a two-dimensional, one-column DataFrame. The optional argument names that column. This does not change the values; it changes the table structure, making the result convenient for display, joining, column selection, or adding more calculated columns.

In [ ]:
country_counts_series = df["Country"].value_counts()
country_counts_frame = country_counts_series.to_frame("TransactionRows")

print("Before to_frame:", type(country_counts_series).__name__, country_counts_series.shape)
print("After to_frame:", type(country_counts_frame).__name__, country_counts_frame.shape)
display(country_counts_series)
country_counts_frame

## 7. Define valid sales

Analytics begins with a clear rule. For purchase reports, keep invoices that do not start with `C` and rows with positive quantity and price. The `.copy()` creates an independent DataFrame. We retain the original `df` because cancellations and damage still matter for quality and loss analysis.

In [ ]:
valid_sales = df.loc[
    ~df["InvoiceNo"].str.startswith("C")
    & df["Quantity"].gt(0)
    & df["UnitPrice"].gt(0)
].copy()

valid_sales

## 8. Group rows by country

`groupby()` forms one group for each country. `agg()` then calculates several measures for every group. `nunique` counts distinct invoices, `sum` adds numeric values, and `size` counts rows. Named aggregation gives the output columns readable names.

In [ ]:
country_summary = (
    valid_sales.groupby("Country")
      .agg(
          InvoiceCount=("InvoiceNo", "nunique"),
          TotalQuantity=("Quantity", "sum"),
          TotalAmount=("Amount", "sum"),
      )
      .sort_values("TotalAmount", ascending=False)
)
country_summary

### What does `observed=True` mean?

`observed` matters when a grouping column has the `category` data type. `observed=True` returns only categories found in the rows; `observed=False` also returns predefined categories that have no rows. For ordinary string columns, such as our original `Country`, it makes no practical difference. Writing it explicitly documents the intended result and avoids relying on a pandas default that may differ between versions.

In [ ]:
category_example = valid_sales.copy()
category_example["Country"] = pd.Categorical(
    category_example["Country"],
    categories=["India", "France", "United Kingdom", "Germany"],
)

all_defined_categories = (
    category_example.groupby("Country", observed=False)["Amount"].sum()
)
categories_present_in_data = (
    category_example.groupby("Country", observed=True)["Amount"].sum()
)

print("observed=False: Germany appears with 0 because it is a defined category")
display(all_defined_categories.to_frame("TotalAmount"))
print("observed=True: only countries present in the data appear")
display(categories_present_in_data.to_frame("TotalAmount"))

## 9. Summarize two categories with `crosstab()`

A cross-tabulation places one category on the rows and another on the columns. By default, `pd.crosstab()` counts matching rows. It is useful for questions such as “how many sales entries did each country have for each SKU?” `margins=True` adds row and column totals named `All`.

In [ ]:
sales_entry_crosstab = pd.crosstab(
    index=valid_sales["Country"],
    columns=valid_sales["StockCode"],
    margins=True,
)
sales_entry_crosstab

The first table counts entries, not physical units. To calculate quantities, supply `values=Quantity` and `aggfunc="sum"`. `fill_value=0` replaces combinations that did not occur. This form behaves like a small pivot table. Compare the two tables: one product entry can contain several units.

In [ ]:
quantity_crosstab = pd.crosstab(
    index=valid_sales["Country"],
    columns=valid_sales["StockCode"],
    values=valid_sales["Quantity"],
    aggfunc="sum",
    margins=True,
).fillna(0).astype(int)
quantity_crosstab

## 10. Combine related tables with `join()`

Real systems often store inventory and supplier details separately. `join()` adds columns by matching an index. Here, `on="SupplierID"` uses that inventory column and matches it against the supplier table's index. A left join keeps every inventory row; unmatched supplier `S9` receives missing values. `validate="many_to_one"` checks that each supplier ID appears at most once in the lookup table.

In [ ]:
inventory = pd.DataFrame([
    {"SKU": "A100", "StockQty": 40, "Price": 5.00, "SupplierID": "S1"},
    {"SKU": "B200", "StockQty": 100, "Price": 1.50, "SupplierID": "S1"},
    {"SKU": "C300", "StockQty": 25, "Price": 7.50, "SupplierID": "S2"},
    {"SKU": "D400", "StockQty": 12, "Price": 20.00, "SupplierID": "S3"},
    {"SKU": "X999", "StockQty": 3, "Price": 0.00, "SupplierID": "S9"},
])

suppliers = pd.DataFrame([
    {"SupplierID": "S1", "Supplier": "WriteWell Ltd"},
    {"SupplierID": "S2", "Supplier": "Kitchen Goods Co"},
    {"SupplierID": "S3", "Supplier": "CarryAll Traders"},
])

display(inventory)
suppliers

In [ ]:
inventory_with_supplier = inventory.join(
    suppliers.set_index("SupplierID"),
    on="SupplierID",
    how="left",
    validate="many_to_one",
)
inventory_with_supplier

Changing `how` changes which keys survive. A left join keeps all five inventory rows, including `X999`. An inner join keeps only rows whose supplier exists in both tables, so `X999` is removed. Use a left join when the inventory list is primary and missing supplier data should remain visible for correction.

In [ ]:
matched_inventory_only = inventory.join(
    suppliers.set_index("SupplierID"),
    on="SupplierID",
    how="inner",
    validate="many_to_one",
)

print(f"Left join rows: {len(inventory_with_supplier)}")
print(f"Inner join rows: {len(matched_inventory_only)}")
matched_inventory_only

## 11. Change the grain: one row per invoice

The raw grain is one product entry. Since an invoice repeats for different products, group by invoice before ranking orders. `TotalItems` uses `size`, so it counts entries regardless of quantity. For example, an invoice containing two product rows has two items here even if those rows contain five physical units.

In [ ]:
invoice_summary = (
    valid_sales.groupby("InvoiceNo")
      .agg(
          TotalAmount=("Amount", "sum"),
          TotalQuantity=("Quantity", "sum"),
          TotalItems=("StockCode", "size"),
      )
      .sort_values("TotalAmount", ascending=False)
)
invoice_summary

## 12. Rank customers by spending

First remove rows without a customer ID, then group and sort customers by total spending. After `reset_index()`, the rows are numbered 0, 1, 2, so adding 1 creates a beginner-friendly rank. This is a simple row-position rank; later you can learn different rules for handling equal values, such as dense ranking.

In [ ]:
customer_ranking = (
    valid_sales.dropna(subset=["CustomerID"])
      .groupby("CustomerID")
      .agg(
          TotalSpent=("Amount", "sum"),
          InvoiceCount=("InvoiceNo", "nunique"),
      )
      .sort_values("TotalSpent", ascending=False)
      .reset_index()
)
customer_ranking.insert(0, "Rank", customer_ranking.index + 1)
customer_ranking

## 13. Find the most sold products

Group by both SKU and description, then add quantities. Ranking by `UnitsSold` answers “which product had the most physical units sold?” Ranking by revenue would answer a different question. Always name the measure behind words such as “top” or “best.”

In [ ]:
product_ranking = (
    valid_sales.groupby(["StockCode", "Description"], as_index=False)
      .agg(
          UnitsSold=("Quantity", "sum"),
          TotalRevenue=("Amount", "sum"),
      )
      .sort_values("UnitsSold", ascending=False)
      .reset_index(drop=True)
)
product_ranking.insert(0, "Rank", product_ranking.index + 1)
product_ranking

## 14. Investigate negative quantities

Do not automatically delete negative quantities. A `C` invoice suggests a cancellation or return; another description may suggest damaged stock. Estimated loss here is reversed selling value, calculated as the positive magnitude of a negative amount. Zero-priced damage has lost units but no measurable monetary loss because product cost is absent.

In [ ]:
negative_qty = df.loc[df["Quantity"].lt(0)].copy()
negative_qty["Reason"] = negative_qty["InvoiceNo"].str.startswith("C").map({
    True: "Cancellation / return",
    False: "Stock adjustment",
})
negative_qty["UnitsRemoved"] = -negative_qty["Quantity"]
negative_qty["EstimatedLoss"] = (-negative_qty["Amount"]).clip(lower=0)

display(negative_qty[["InvoiceNo", "Description", "Reason", "UnitsRemoved", "EstimatedLoss"]])
print(f"Total estimated loss: {negative_qty['EstimatedLoss'].sum():.2f}")

## 15. Inspect memory and non-null values with `info()`

`info()` prints a compact structural summary: row index, column names, non-null counts, data types, and memory usage. `memory_usage="deep"` inspects the objects stored inside text columns, giving a more realistic estimate than counting only references. It is more accurate but can take longer on very large data. `info()` prints output and returns `None`.

In [ ]:
df.info(memory_usage="deep")

## 16. Control display with `pd.set_option()`

Pandas may shorten wide or long output. `pd.set_option()` changes how results are displayed; it does not alter the stored data. Options remain active for the Python session, so use `pd.reset_option()` when a temporary format is no longer needed. Common options control visible columns, rows, column width, and floating-point formatting.

In [ ]:
pd.set_option("display.max_columns", None)       # Show every column
pd.set_option("display.max_colwidth", 40)       # Limit long text display
pd.set_option("display.float_format", "{:,.2f}".format)

display(df.head(2))

# Restore only the custom float format; max_columns remains useful here.
pd.reset_option("display.float_format")

## 17. Inspect the last rows with `tail()`

`tail()` returns the last five rows by default; `tail(3)` returns the last three. It is useful for detecting footer records, incomplete final rows, adjustments appended at the end, or checking the bottom of a sorted result. “Last” means current row order—`tail()` does not sort the data.

In [ ]:
print("Last three raw rows:")
display(df.tail(3))

print("Two lowest-spending customers after sorting high to low:")
customer_ranking.tail(2)

## 18. Combine Boolean results with `any()` and `all()`

`any()` is True when at least one value is True; `all()` is True only when every value is True. On a DataFrame, `axis=1` checks across columns for each row, while the default `axis=0` checks down rows for each column. These methods are useful for quality flags, filters, and validation rules.

In [ ]:
quality_flags = pd.DataFrame({
    "MissingCustomer": df["CustomerID"].isna(),
    "NonPositiveQuantity": df["Quantity"].le(0),
    "NonPositivePrice": df["UnitPrice"].le(0),
})

quality_flags["HasAnyIssue"] = quality_flags.any(axis=1)
quality_flags["HasEveryIssue"] = quality_flags.iloc[:, :3].all(axis=1)
display(quality_flags)

print("Does each issue occur anywhere?")
display(quality_flags.iloc[:, :3].any().to_frame("OccursAnywhere"))

A common validation pattern applies `all()` to one Boolean Series. The expression below checks every row in `valid_sales`. If even one quantity is zero or negative, the result becomes False. An `assert` can stop execution when a required rule fails, but first printing the Boolean is easier while learning.

In [ ]:
all_quantities_are_positive = valid_sales["Quantity"].gt(0).all()
all_prices_are_positive = valid_sales["UnitPrice"].gt(0).all()

print("All quantities positive:", all_quantities_are_positive)
print("All prices positive:", all_prices_are_positive)
assert all_quantities_are_positive and all_prices_are_positive

## 19. Understand distributions with `quantile()`

A quantile is a boundary below which a proportion of observations falls. The 0.25, 0.50, and 0.75 quantiles are Q1, median, and Q3. For example, a median quantity of 2 means at least half the recorded quantities are no greater than 2. Quantiles are less influenced by extreme values than the mean and help identify spread and possible outliers.

In [ ]:
quantity_quantiles = valid_sales["Quantity"].quantile(
    [0.00, 0.25, 0.50, 0.75, 1.00]
).rename(index={
    0.00: "Minimum", 0.25: "Q1", 0.50: "Median",
    0.75: "Q3", 1.00: "Maximum",
})
quantity_quantiles.to_frame("Quantity")

The interquartile range (IQR) is `Q3 - Q1`, covering the middle 50% of values. A common screening rule marks values above `Q3 + 1.5 × IQR` or below `Q1 - 1.5 × IQR`. Such values are unusual, not automatically incorrect; a large wholesale order may be valid.

In [ ]:
q1 = valid_sales["Quantity"].quantile(0.25)
q3 = valid_sales["Quantity"].quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}, upper fence: {upper_fence}")
valid_sales.loc[valid_sales["Quantity"] > upper_fence]

## 20. Limit values with `clip()`

`clip()` replaces values outside specified boundaries. Values below `lower` become the lower boundary, values above `upper` become the upper boundary, and values inside the range remain unchanged. Clipping does not remove rows and does not prove that extreme values are errors. It is useful for enforcing known limits, preparing visualizations, or converting negative reversed amounts into non-negative loss magnitudes.

In [ ]:
clip_example = pd.Series(
    [-5, 2, 10, 100],
    index=["Negative", "Small", "Normal", "Extreme"],
    name="Original",
)

clip_comparison = pd.DataFrame({
    "Original": clip_example,
    "LowerOnly_0": clip_example.clip(lower=0),
    "UpperOnly_20": clip_example.clip(upper=20),
    "Between_0_and_20": clip_example.clip(lower=0, upper=20),
})
clip_comparison

In the loss calculation used earlier, `(-Amount).clip(lower=0)` first reverses the sign and then replaces negative results with zero. Therefore, a return amount of `-7.50` becomes an estimated loss of `7.50`, while a positive sale does not become a negative loss. Keep the original amount column so no source information is destroyed.

In [ ]:
loss_clip_example = df[["InvoiceNo", "Amount"]].copy()
loss_clip_example["EstimatedLoss"] = (-loss_clip_example["Amount"]).clip(lower=0)
loss_clip_example.loc[loss_clip_example["EstimatedLoss"].gt(0)]

## 21. Reshape rows into columns with `pivot()`

Long data stores one observation per row. `pivot()` turns unique values from one column into new column headings. Here, each region becomes a row, each SKU becomes a column, and units become cell values. `pivot()` requires each region-SKU combination to be unique; when duplicates need aggregation, use `pivot_table()` instead.

In [ ]:
long_product_sales = pd.DataFrame([
    {"Region": "North", "SKU": "A100", "Units": 10},
    {"Region": "North", "SKU": "B200", "Units": 6},
    {"Region": "North", "SKU": "C300", "Units": 4},
    {"Region": "South", "SKU": "A100", "Units": 8},
    {"Region": "South", "SKU": "B200", "Units": 9},
    {"Region": "South", "SKU": "C300", "Units": 5},
])

sales_wide = long_product_sales.pivot(
    index="Region", columns="SKU", values="Units"
)

display(long_product_sales)
sales_wide

### Convert columns back into rows with `melt()`

`melt()` reverses a wide table into long form. `id_vars` identifies columns that remain fixed. The other columns are gathered into a variable column and a value column. Long form is commonly preferred for grouping and plotting because the measured variable has one consistent column.

In [ ]:
sales_long_again = (
    sales_wide.reset_index()
      .melt(
          id_vars="Region",
          var_name="SKU",
          value_name="Units",
      )
      .sort_values(["Region", "SKU"])
      .reset_index(drop=True)
)
sales_long_again

### Literally swap rows and columns with `.T`

The transpose property `.T` swaps the complete row and column axes. Unlike `pivot()`, it does not use values from a particular column as headings according to a business rule. It is useful for small display tables, but mixed original column types may become a general object type after transposing.

In [ ]:
inventory_for_display = inventory.set_index("SKU")[["StockQty", "Price"]]
display(inventory_for_display)
inventory_for_display.T

## 22. Expand list values into rows with `explode()`

Sometimes one cell contains a list. `explode()` creates one row for each list element and repeats the other column values. This converts nested data into a tabular form suitable for grouping. When exploding multiple columns together, their lists must have matching lengths so corresponding SKU and quantity values remain paired.

In [ ]:
packed_orders = pd.DataFrame([
    {"InvoiceNo": "E1", "SKUs": ["A100", "B200"], "Quantities": [2, 3]},
    {"InvoiceNo": "E2", "SKUs": ["C300"], "Quantities": [1]},
    {"InvoiceNo": "E3", "SKUs": ["A100", "C300", "D400"], "Quantities": [1, 2, 1]},
])

order_lines = packed_orders.explode(
    ["SKUs", "Quantities"], ignore_index=True
)

display(packed_orders)
order_lines

After exploding, invoice `E1` occupies two rows and `E3` occupies three. `ignore_index=True` creates a clean 0-based index. The exploded values can retain an object data type, so convert them when numeric operations are needed—for example, `order_lines["Quantities"] = order_lines["Quantities"].astype("int64")`.

In [ ]:
order_lines["Quantities"] = order_lines["Quantities"].astype("int64")
order_lines.groupby("SKUs")["Quantities"].sum().to_frame("TotalQuantity")

## Recap

A reliable analysis follows a sequence: understand the row grain, inspect types and missing values, define valid records, derive meaningful columns, group at the level needed by the question, aggregate, sort, and interpret. The same operations scale from these 12 rows to the larger e-commerce file; only the number of rows changes.